# NB05 — RoPE & Embedding Kernels

**North star callback:** Rotary position embeddings (RoPE) are applied to Q and K before
every attention layer. In HuggingFace, this is two separate ops (two HBM round-trips).
Unsloth fuses them into one kernel. This notebook builds the fused version.

## 1. RoPE math

In [1]:
import torch, math

def precompute_freqs(d: int, max_seq: int, theta: float = 10000.0, device="cuda"):
    """Precompute cos/sin tables for rotary embeddings."""
    freqs = 1.0 / (theta ** (torch.arange(0, d, 2, device=device).float() / d))
    t = torch.arange(max_seq, device=device)
    freqs = torch.outer(t, freqs)  # (max_seq, d//2)
    return torch.cos(freqs), torch.sin(freqs)

def apply_rope_unfused(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
    """Reference RoPE — two separate ops (matches HuggingFace rotate_half format)."""
    d = x.shape[-1]
    x1 = x[..., :d//2]
    x2 = x[..., d//2:]
    rotated = torch.cat([-x2, x1], dim=-1)
    return x * cos + rotated * sin

B, H, N, d = 2, 32, 512, 128
cos_half, sin_half = precompute_freqs(d, N)   # (N, d//2) — the raw half-dim tables

# HuggingFace duplicates cos/sin to full head_dim: [cos, cos] and [sin, sin]
cos_hf = torch.cat([cos_half, cos_half], dim=-1)[None, None, :, :].expand(B, H, N, d)
sin_hf = torch.cat([sin_half, sin_half], dim=-1)[None, None, :, :].expand(B, H, N, d)

Q = torch.randn(B, H, N, d, device="cuda", dtype=torch.bfloat16)
Q_rope_ref = apply_rope_unfused(Q.float(), cos_hf.float(), sin_hf.float()).bfloat16()
print("Reference RoPE shape:", Q_rope_ref.shape)
print("cos_half shape:", cos_half.shape, "  sin_half shape:", sin_half.shape)

Reference RoPE shape: torch.Size([2, 32, 512, 128])
cos_half shape: torch.Size([512, 64])   sin_half shape: torch.Size([512, 64])


## 2. Unfused bandwidth

In [2]:
import triton

def hf_rope(x):
    x1, x2 = x[..., :d//2], x[..., d//2:]
    rotated = torch.cat([-x2, x1], dim=-1)
    return x * cos_hf + rotated * sin_hf

ms_unfused = triton.testing.do_bench(lambda: hf_rope(Q))
# Reads: Q (bf16), cos_hf (fp32), sin_hf (fp32); Write: out (bf16)
q_bytes     = Q.numel() * Q.element_size()           # bf16 read
cos_bytes   = cos_hf.numel() * 4                     # fp32 read (expanded)
sin_bytes   = sin_hf.numel() * 4
out_bytes   = Q.numel() * Q.element_size()           # bf16 write
bytes_unfused = q_bytes + cos_bytes + sin_bytes + out_bytes
print(f"Unfused RoPE : {ms_unfused:.3f} ms  "
      f"({bytes_unfused / ms_unfused * 1e-6:.0f} GB/s effective)")
print(f"  Tensor bytes: Q={q_bytes/1e6:.1f}MB  cos={cos_bytes/1e6:.1f}MB  "
      f"sin={sin_bytes/1e6:.1f}MB  out={out_bytes/1e6:.1f}MB")

Unfused RoPE : 0.093 ms  (539 GB/s effective)
  Tensor bytes: Q=8.4MB  cos=16.8MB  sin=16.8MB  out=8.4MB


## 3. Fused RoPE kernel in Triton

The key insight: instead of loading Q, cos, and sin in separate ops, we load all three
in one kernel pass and write the result immediately — one HBM round-trip instead of two.

The kernel processes `BLOCK_D = d//2` elements per program. Each program handles one
sequence position and one (batch × head) pair. It loads the first and second halves of
Q, then applies the rotate-half formula:

```
out[:half_d] = q1 * cos - q2 * sin
out[half_d:] = q2 * cos + q1 * sin
```

In [3]:
import triton.language as tl

@triton.jit
def rope_fused_kernel(
    Q_ptr, Cos_ptr, Sin_ptr, Out_ptr,
    stride_qbh, stride_qn,          # strides for (B*H) and N dims; d is always contiguous
    head_dim,
    BLOCK_D: tl.constexpr,          # must equal head_dim // 2 (a power of 2)
):
    """Fused RoPE: load Q + cos + sin once, write rotated output in one pass."""
    pid_n  = tl.program_id(0)   # sequence position
    pid_bh = tl.program_id(1)   # flattened (batch * heads) index

    half_d = head_dim // 2
    offs = tl.arange(0, BLOCK_D)   # 0..half_d-1
    mask = offs < half_d

    base = pid_bh * stride_qbh + pid_n * stride_qn

    # Load first half and second half of q for this (bh, n) pair
    q1 = tl.load(Q_ptr + base + offs,          mask=mask, other=0.0).to(tl.float32)
    q2 = tl.load(Q_ptr + base + offs + half_d, mask=mask, other=0.0).to(tl.float32)

    # Load cos/sin: indexed by (position, d//2) — shared across batch and heads
    cs_offs = pid_n * half_d + offs
    cos_v = tl.load(Cos_ptr + cs_offs, mask=mask, other=0.0)
    sin_v = tl.load(Sin_ptr + cs_offs, mask=mask, other=0.0)

    # Rotate-half: matches HuggingFace apply_rotary_pos_emb with rotate_half format
    out1 = q1 * cos_v - q2 * sin_v
    out2 = q2 * cos_v + q1 * sin_v

    tl.store(Out_ptr + base + offs,          out1.to(tl.bfloat16), mask=mask)
    tl.store(Out_ptr + base + offs + half_d, out2.to(tl.bfloat16), mask=mask)


def apply_rope_fused(Q: torch.Tensor, cos_half: torch.Tensor, sin_half: torch.Tensor):
    """Launch the fused RoPE kernel. cos_half/sin_half are shape (N, d//2)."""
    B, H, N, d = Q.shape
    Out = torch.empty_like(Q)
    BLOCK_D = triton.next_power_of_2(d // 2)

    Q_f   = Q.reshape(B * H, N, d).contiguous()
    Out_f = Out.reshape(B * H, N, d)

    grid = (N, B * H)
    rope_fused_kernel[grid](
        Q_f, cos_half.contiguous(), sin_half.contiguous(), Out_f,
        Q_f.stride(0), Q_f.stride(1),   # stride_qbh, stride_qn
        d,
        BLOCK_D=BLOCK_D,
    )
    return Out_f.reshape(B, H, N, d)


# Correctness check
# Threshold is 0.02: bf16 has 7-bit mantissa so for values ~2.0 the
# quantization step is 2^(1-7) = 0.015625 — one ULP of difference is expected.
Q_rope_fused = apply_rope_fused(Q, cos_half.float(), sin_half.float())
max_err = (Q_rope_fused.float() - Q_rope_ref.float()).abs().max().item()
print(f"Max error vs reference: {max_err:.6f}  ", end="")
print("✓" if max_err < 0.02 else "✗ FAIL")

Max error vs reference: 0.015625  ✓


## 4. Bandwidth comparison

In [4]:
ms_fused = triton.testing.do_bench(
    lambda: apply_rope_fused(Q, cos_half.float(), sin_half.float())
)
# Fused reads: Q (bf16), cos_half (fp32), sin_half (fp32); writes: out (bf16)
# cos_half/sin_half are (N, d//2) — no batch/head duplication needed
cos_half_bytes = cos_half.numel() * 4
sin_half_bytes = sin_half.numel() * 4
bytes_fused = q_bytes + cos_half_bytes + sin_half_bytes + out_bytes

print(f"Unfused RoPE : {ms_unfused:.3f} ms  ({bytes_unfused / ms_unfused * 1e-6:.0f} GB/s)")
print(f"Fused RoPE   : {ms_fused:.3f} ms  ({bytes_fused / ms_fused * 1e-6:.0f} GB/s)")
print(f"Speedup      : {ms_unfused / ms_fused:.2f}x")
print()
print(f"HBM saved by fusion: "
      f"{(bytes_unfused - bytes_fused) / 1e6:.1f} MB per Q application")
print(f"  (unfused loads expanded cos/sin {cos_hf.numel()*4/1e6:.1f} MB, "
      f"fused loads compact cos/sin {(cos_half_bytes+sin_half_bytes)/1e6:.1f} MB)")

Unfused RoPE : 0.093 ms  (539 GB/s)
Fused RoPE   : 0.027 ms  (624 GB/s)
Speedup      : 3.42x

HBM saved by fusion: 33.3 MB per Q application
  (unfused loads expanded cos/sin 16.8 MB, fused loads compact cos/sin 0.3 MB)


## 5. Read Unsloth's RoPE implementation

Unsloth's fused RoPE is in `unsloth/kernels/rope_embedding.py`.
Key things to look for:
- How they handle the interleaved vs rotary-half formats
- The autotuned block sizes
- How they integrate into the attention forward pass (inlined, not a separate call)

In [5]:
import sys, inspect
sys.path.insert(0, "../unsloth")
from unsloth.kernels import rope_embedding
print(inspect.getsource(rope_embedding))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


# Copyright 2023-present Daniel Han-Chen, Michael Han-Chen & the Unsloth team. All rights reserved.
#
# This program is free software: you can redistribute it and/or modify
# it under the terms of the GNU Lesser General Public License as published by
# the Free Software Foundation, either version 3 of the License, or
# (at your option) any later version.
#
# This program is distributed in the hope that it will be useful,
# but WITHOUT ANY WARRANTY; without even the implied warranty of
# MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
# GNU General Public License for more details.
#
# You should have received a copy of the GNU Lesser General Public License
# along with this program.  If not, see <https://www.gnu.org/licenses/>.

import triton
import triton.language as tl
import torch
from ..device_type import DEVICE_COUNT
from .utils import calculate_settings, torch_gpu_device, torch_device_stream


def _rope_embedding_QK(
    Q,
    Q_batch_stride,
    Q_head_stride,
    

## 6. Exercises

1. **Measure the bandwidth saving**: The fused kernel loads compact `(N, d//2)` cos/sin
   tables. The unfused path uses expanded `(B, H, N, d)` tensors. For a 70B model
   with B=2, H=64, N=4096, d=128 — what is the total HBM saved per transformer layer?

2. **Extend to Q+K**: Launch the same kernel for K simultaneously. How would you
   modify `apply_rope_fused` to process both Q and K in a single kernel launch?
   What is the theoretical bandwidth reduction compared to two separate calls?

3. **Add autotuning**: Wrap `rope_fused_kernel` with `@triton.autotune` to search
   over different `BLOCK_D` values and `num_warps`. Does it improve throughput on the
   RTX 4090 for d=128?